In [7]:
import numpy as np
import pandas as pd
from src.features.process_features import *
pd.set_option('display.max_columns', None)

# 0. Load dataframe

We start by removing `SP` (was processed and is not used anymore), `winnerOr2ndName`, `winnerOr2ndId` (those twoa re not used later)

In [8]:
# Specify the columns you want to remove here
cols_to_remove = ["SP","winnerOr2ndName","winnerOr2ndId"] 
all_cols = ['SP', 'resultPosition', 
               'resultBtnDistance', 'resultSectionalTime',
               'resultComment', 'resultRunTime', 
               'resultDogWeight', 'winnerOr2ndName',
               'winnerOr2ndId', 'resultAdjustedTime', 
               'trapNumber', 'raceTime',
               'raceDate', 'raceId', 
               'raceType', 'raceClass', 
               'raceDistance','raceGoing', 
               'raceWinTime', 'meetingId', 
               'trackName', 'dogId',
               'numeratorSP', 'denominatorSP', 
               'commentScore','resultSectionalTime_missing']
cols_to_use = [col for col in all_cols if col not in cols_to_remove]

dog_infos = pd.read_csv("../data/intermediate/04_dog_infos_cleaned.csv", usecols=cols_to_use)

In [9]:
dog_infos.head()

,resultPosition,resultBtnDistance,resultSectionalTime,resultComment,resultRunTime,resultDogWeight,resultAdjustedTime,trapNumber,raceTime,raceDate,raceId,raceType,raceClass,raceDistance,raceGoing,raceWinTime,meetingId,trackName,dogId,numeratorSP,denominatorSP,commentScore,resultSectionalTime_missing
0,4.0,2.00,4.46,"Middle,EveryChance",28.73,34.1,28.73,5,21:18:00,2022-03-24,826826,Flat,A1,480.0,0.0,28.24,383435,Monmore,100211,5.0,4.000000,-1.0,0
1,3.0,1.75,4.46,"CrdStt,Middle,RanOn",28.46,33.9,28.36,3,20:43:00,2022-03-12,823113,Flat,OR,480.0,-10.0,28.06,382921,Monmore,100211,7.0,1.000000,0.0,0
2,3.0,1.00,4.53,"Mid,FcdTCk&Crd1",28.72,33.9,28.72,3,16:48:00,2022-03-04,821053,Flat,A1,480.0,0.0,28.58,382798,Monmore,100211,11.0,1.675319,-1.0,0
3,6.0,10.00,4.45,"Crd1,Chl&Crd&FellRunIn",28.98,34.1,28.85,4,18:17:00,2022-01-29,811419,Flat,A1,480.0,0.0,28.51,381677,Monmore,100211,1.0,1.000000,0.0,0
4,2.0,10.00,4.45,"CrdStt,EP,Mid,RanOn",28.69,34.5,28.49,5,16:08:00,2022-01-21,809513,Flat,A1,480.0,-20.0,28.63,381477,Monmore,100211,11.0,1.675319,1.0,0


# 1. Feature Engineering

## 1.1 Split sprint/A class races

We start by separating sprint and A class races to use in the model. The idea is to compare what is comparable and sprint races are a different kind of races.

In [10]:
CLASS_CONFIG = {

    # ── A GRADES: directly in LSTM sequence ──────────────────────
    # Numeric grade = inverse of number (A1 = best = highest level)
    'A1':  {'group': 'graded_a', 'relevance': 1.0, 'quality': 13, 'in_lstm': True},
    'A2':  {'group': 'graded_a', 'relevance': 1.0, 'quality': 12, 'in_lstm': True},
    'A3':  {'group': 'graded_a', 'relevance': 1.0, 'quality': 11, 'in_lstm': True},
    'A4':  {'group': 'graded_a', 'relevance': 1.0, 'quality': 10, 'in_lstm': True},
    'A5':  {'group': 'graded_a', 'relevance': 1.0, 'quality':  9, 'in_lstm': True},
    'A6':  {'group': 'graded_a', 'relevance': 1.0, 'quality':  8, 'in_lstm': True},
    'A7':  {'group': 'graded_a', 'relevance': 1.0, 'quality':  7, 'in_lstm': True},
    'A8':  {'group': 'graded_a', 'relevance': 1.0, 'quality':  6, 'in_lstm': True},
    'A9':  {'group': 'graded_a', 'relevance': 1.0, 'quality':  5, 'in_lstm': True},
    'A10': {'group': 'graded_a', 'relevance': 1.0, 'quality':  4, 'in_lstm': True},
    'A11': {'group': 'graded_a', 'relevance': 1.0, 'quality':  3, 'in_lstm': True},
    'A12': {'group': 'graded_a', 'relevance': 1.0, 'quality':  2, 'in_lstm': True},
    'A15': {'group': 'graded_a', 'relevance': 1.0, 'quality':  1, 'in_lstm': True},

    # ── OPEN RACES: static features, quality above A1 ───────────
    'OR':  {'group': 'open', 'relevance': 0.9, 'quality': 16, 'in_lstm': False},
    'OR1': {'group': 'open', 'relevance': 0.9, 'quality': 15, 'in_lstm': False},
    'OR2': {'group': 'open', 'relevance': 0.85,'quality': 14, 'in_lstm': False},
    'OR3': {'group': 'open', 'relevance': 0.80,'quality': 13, 'in_lstm': False},
    'GR':  {'group': 'open', 'relevance': 0.9, 'quality': 15, 'in_lstm': False},
    'E1':  {'group': 'open', 'relevance': 0.85,'quality': 14, 'in_lstm': False},
    'E2':  {'group': 'open', 'relevance': 0.80,'quality': 13, 'in_lstm': False},
    'E3':  {'group': 'open', 'relevance': 0.75,'quality': 12, 'in_lstm': False},

    # ── B GRADES: static only, sex-restricted ───────────────────
    'B1':  {'group': 'graded_b', 'relevance': 0.75,'quality': 10, 'in_lstm': False},
    'B2':  {'group': 'graded_b', 'relevance': 0.75,'quality':  9, 'in_lstm': False},
    'B3':  {'group': 'graded_b', 'relevance': 0.75,'quality':  8, 'in_lstm': False},
    'B4':  {'group': 'graded_b', 'relevance': 0.75,'quality':  7, 'in_lstm': False},
    'B5':  {'group': 'graded_b', 'relevance': 0.75,'quality':  6, 'in_lstm': False},
    'B6':  {'group': 'graded_b', 'relevance': 0.75,'quality':  5, 'in_lstm': False},
    'B7':  {'group': 'graded_b', 'relevance': 0.75,'quality':  4, 'in_lstm': False},
    'B8':  {'group': 'graded_b', 'relevance': 0.75,'quality':  3, 'in_lstm': False},
    'B9':  {'group': 'graded_b', 'relevance': 0.75,'quality':  2, 'in_lstm': False},
    'B15': {'group': 'graded_b', 'relevance': 0.75,'quality':  1, 'in_lstm': False},

    # ── S GRADES: sprint — static only, incompatible times ──────
    'S1':  {'group': 'sprint', 'relevance': 0.55,'quality':  8, 'in_lstm': False},
    'S2':  {'group': 'sprint', 'relevance': 0.55,'quality':  7, 'in_lstm': False},
    'S3':  {'group': 'sprint', 'relevance': 0.55,'quality':  6, 'in_lstm': False},
    'S4':  {'group': 'sprint', 'relevance': 0.55,'quality':  5, 'in_lstm': False},
    'S5':  {'group': 'sprint', 'relevance': 0.55,'quality':  4, 'in_lstm': False},
    'S6':  {'group': 'sprint', 'relevance': 0.55,'quality':  3, 'in_lstm': False},
    'S7':  {'group': 'sprint', 'relevance': 0.55,'quality':  2, 'in_lstm': False},
    'S8':  {'group': 'sprint', 'relevance': 0.55,'quality':  1, 'in_lstm': False},
    'S15': {'group': 'sprint', 'relevance': 0.55,'quality':  1, 'in_lstm': False},

    # ── D GRADES: distance — static only, incompatible times ────
    'D1':  {'group': 'distance', 'relevance': 0.45,'quality':  5, 'in_lstm': False},
    'D2':  {'group': 'distance', 'relevance': 0.45,'quality':  4, 'in_lstm': False},
    'D3':  {'group': 'distance', 'relevance': 0.45,'quality':  3, 'in_lstm': False},
    'D4':  {'group': 'distance', 'relevance': 0.45,'quality':  2, 'in_lstm': False},
    'D5':  {'group': 'distance', 'relevance': 0.45,'quality':  1, 'in_lstm': False},

    # ── M GRADES: middle distance — static only ──────────────────
    'M1':  {'group': 'middle', 'relevance': 0.60,'quality':  5, 'in_lstm': False},
    'M2':  {'group': 'middle', 'relevance': 0.60,'quality':  4, 'in_lstm': False},
    'M3':  {'group': 'middle', 'relevance': 0.60,'quality':  3, 'in_lstm': False},
    'M4':  {'group': 'middle', 'relevance': 0.60,'quality':  2, 'in_lstm': False},
    'M5':  {'group': 'middle', 'relevance': 0.60,'quality':  1, 'in_lstm': False},

    # ── P GRADES: puppy/juvenile — static only ───────────────────
    'P1':  {'group': 'puppy', 'relevance': 0.40,'quality':  9, 'in_lstm': False},
    'P2':  {'group': 'puppy', 'relevance': 0.40,'quality':  8, 'in_lstm': False},
    'P3':  {'group': 'puppy', 'relevance': 0.40,'quality':  7, 'in_lstm': False},
    'P4':  {'group': 'puppy', 'relevance': 0.40,'quality':  6, 'in_lstm': False},
    'P5':  {'group': 'puppy', 'relevance': 0.40,'quality':  5, 'in_lstm': False},
    'P6':  {'group': 'puppy', 'relevance': 0.40,'quality':  4, 'in_lstm': False},
    'P7':  {'group': 'puppy', 'relevance': 0.40,'quality':  3, 'in_lstm': False},
    'P8':  {'group': 'puppy', 'relevance': 0.40,'quality':  2, 'in_lstm': False},
    'P9':  {'group': 'puppy', 'relevance': 0.40,'quality':  1, 'in_lstm': False},
    'P10': {'group': 'puppy', 'relevance': 0.40,'quality':  1, 'in_lstm': False},

    # ── HURDLES: exclude entirely from performance features ──────
    'H1':  {'group': 'hurdles', 'relevance': 0.05,'quality': 0, 'in_lstm': False},
    'H2':  {'group': 'hurdles', 'relevance': 0.05,'quality': 0, 'in_lstm': False},
    'H3':  {'group': 'hurdles', 'relevance': 0.05,'quality': 0, 'in_lstm': False},
    'H4':  {'group': 'hurdles', 'relevance': 0.05,'quality': 0, 'in_lstm': False},
    'HS1': {'group': 'hurdles', 'relevance': 0.05,'quality': 0, 'in_lstm': False},
    'HS2': {'group': 'hurdles', 'relevance': 0.05,'quality': 0, 'in_lstm': False},
    'HD2': {'group': 'hurdles', 'relevance': 0.05,'quality': 0, 'in_lstm': False},
    'HP':  {'group': 'hurdles', 'relevance': 0.05,'quality': 0, 'in_lstm': False},

    # ── TRIALS / SPECIAL: flag only ──────────────────────────────
    'IT':  {'group': 'trial', 'relevance': 0.05,'quality': 0, 'in_lstm': False},
    'IV':  {'group': 'trial', 'relevance': 0.05,'quality': 0, 'in_lstm': False},
    'KS':  {'group': 'trial', 'relevance': 0.05,'quality': 0, 'in_lstm': False},
    'ks':  {'group': 'trial', 'relevance': 0.05,'quality': 0, 'in_lstm': False},
}

FALLBACK = {'group': 'unknown', 'relevance': 0.10, 'quality': 0, 'in_lstm': False}

def get_class_config(race_class):
    if pd.isna(race_class):
        return FALLBACK
    return CLASS_CONFIG.get(str(race_class).strip(), FALLBACK)

In [11]:
def process_dog_history(dog_history):
    """
    Route each race in history to the right processing stream.
    """
    
    h = dog_history.copy()
    h['classGroup']   = h['raceClass'].apply(lambda c: get_class_config(c)['group'])
    h['relevance']     = h['raceClass'].apply(lambda c: get_class_config(c)['relevance'])
    h['qualityLevel'] = h['raceClass'].apply(lambda c: get_class_config(c)['quality'])
    h['inLSTM']       = h['raceClass'].apply(lambda c: get_class_config(c)['in_lstm'])

    # ── Stream 1: LSTM sequence ───────────────────────────────────
    # Only A-grade races — coherent times, directly comparable
    lstm_seq = (
        h[h['in_lstm']]
        .sort_values('raceDate', ascending=False)
        .head(10)
    )

    # ── Stream 2: Static performance features ─────────────────────
    # All classes weighted by relevance, excluding hurdles/trials
    perf_history = h[h['relevance'] > 0.10]

    # ── Stream 3: Binary flags ────────────────────────────────────
    # Signals derived from presence/absence of specific class types
    flags = _compute_flags(h)

    return lstm_seq, perf_history, flags


def _compute_flags(h):
    today = h['raceDate'].max()

    return {
        # Class trajectory: where has the dog been racing recently?
        'class_trajectory_5r': (
            h.sort_values('raceDate', ascending=False)
            .head(5)['quality_level'].mean() - 9.0  # 9 ≈ mid A-grade
        ),

        # Immediate class drop/rise into target A race
        'last_race_quality': (
            h.sort_values('raceDate', ascending=False)
            .iloc[0]['quality_level'] if len(h) > 0 else 9
        ),

        # Has ever competed in open/featured race
        'has_open_race_exp': int(
            h['class_group'].isin(['open']).any()
        ),
        'won_open_race': int(
            ((h['class_group'] == 'open') & (h['resultPosition'] == 1)).any()
        ),

        # Sprint experience — proxy for early pace ability
        'sprint_win_rate': (
            h[h['class_group'] == 'sprint']['resultPosition'].mean()
            if (h['class_group'] == 'sprint').sum() >= 3 else np.nan
        ),

        # Trial in last 14 days — trainer preparation signal
        'had_recent_trial': int((
            (h['class_group'].isin(['trial'])) &
            (h['raceDate'] >= today - pd.Timedelta(days=14))
        ).any()),

        # Hurdles contamination — how much of history is useless
        'hurdles_history_pct': (
            (h['class_group'] == 'hurdles').mean()
        ),

        # History quality — average relevance of past races
        'history_comparability': h['relevance'].mean(),

        # KS/ks — normalise case first, then flag
        # These appear to be a specific competition type
        'has_ks_race': int(
            h['race_class'].str.upper().eq('KS').any()
            if h['race_class'].notna().any() else False
        ),
    }

In [12]:
#################### APPLY HERE _compute_flags TO DOG_INFOS ####################
dog_infos["raceDate"] = pd.to_datetime(dog_infos["raceDate"], errors="coerce")
dog_infos["raceClass"] = dog_infos["raceClass"].astype("string").str.strip().str.upper()

flag_feature_names = [
    "class_trajectory_5r",
    "last_race_quality",
    "has_open_race_exp",
    "won_open_race",
    "sprint_win_rate",
    "had_recent_trial",
    "hurdles_history_pct",
    "history_comparability",
    "has_ks_race",
]

def _empty_history_flags():
    return {
        "class_trajectory_5r": np.nan,
        "last_race_quality": 9,
        "has_open_race_exp": 0,
        "won_open_race": 0,
        "sprint_win_rate": np.nan,
        "had_recent_trial": 0,
        "hurdles_history_pct": 0.0,
        "history_comparability": np.nan,
        "has_ks_race": 0,
    }

def _apply_compute_flags_to_group(dog_history):
    dog_history = dog_history.sort_values(["raceDate", "raceId"]).copy()

    configs = dog_history["raceClass"].apply(get_class_config)
    dog_history["class_group"] = configs.apply(lambda cfg: cfg["group"])
    dog_history["relevance"] = configs.apply(lambda cfg: cfg["relevance"])
    dog_history["quality_level"] = configs.apply(lambda cfg: cfg["quality"])
    dog_history["race_class"] = dog_history["raceClass"]

    history_flags = []
    recent_qualities = []
    sprint_positions = []
    trial_dates = []

    total_past_races = 0
    hurdles_count = 0
    relevance_sum = 0.0
    has_open_race_exp = False
    won_open_race = False
    has_ks_race = False
    last_race_quality = 9

    for row in dog_history.itertuples():
        if total_past_races == 0:
            history_flags.append(_empty_history_flags())
        else:
            current_date = row.raceDate
            if pd.isna(current_date):
                had_recent_trial = 0
            else:
                cutoff_date = current_date - pd.Timedelta(days=14)
                trial_dates = [d for d in trial_dates if d >= cutoff_date]
                had_recent_trial = int(len(trial_dates) > 0)

            history_flags.append({
                "class_trajectory_5r": np.mean(recent_qualities) - 9.0 if recent_qualities else np.nan,
                "last_race_quality": last_race_quality,
                "has_open_race_exp": int(has_open_race_exp),
                "won_open_race": int(won_open_race),
                "sprint_win_rate": np.mean(sprint_positions) if len(sprint_positions) >= 3 else np.nan,
                "had_recent_trial": had_recent_trial,
                "hurdles_history_pct": hurdles_count / total_past_races,
                "history_comparability": relevance_sum / total_past_races,
                "has_ks_race": int(has_ks_race),
            })

        total_past_races += 1
        relevance_sum += row.relevance
        last_race_quality = row.quality_level

        recent_qualities.append(row.quality_level)
        if len(recent_qualities) > 5:
            recent_qualities.pop(0)

        if row.class_group == "open":
            has_open_race_exp = True
            won_open_race = won_open_race or (row.resultPosition == 1)

        if row.class_group == "sprint" and pd.notna(row.resultPosition):
            sprint_positions.append(row.resultPosition)

        if row.class_group == "trial" and pd.notna(row.raceDate):
            trial_dates.append(row.raceDate)

        if row.class_group == "hurdles":
            hurdles_count += 1

        race_class_value = str(row.race_class).upper() if pd.notna(row.race_class) else ""
        if race_class_value == "KS":
            has_ks_race = True

    return pd.DataFrame(history_flags, index=dog_history.index)[flag_feature_names]

dog_history_flags = (
    dog_infos.sort_values(["dogId", "raceDate", "raceId"])
    .groupby("dogId", group_keys=False)
    .apply(_apply_compute_flags_to_group)
    .reindex(dog_infos.index)
)

dog_infos = pd.concat([dog_infos, dog_history_flags], axis=1)
dog_infos[flag_feature_names].head()
################################################################################

,class_trajectory_5r,last_race_quality,has_open_race_exp,won_open_race,sprint_win_rate,had_recent_trial,hurdles_history_pct,history_comparability,has_ks_race
0,5.2,16,1,1,NaN,0,0.0,0.920000,0
1,5.2,13,1,1,NaN,0,0.0,0.921429,0
2,5.8,13,1,1,NaN,0,0.0,0.915385,0
3,6.4,13,1,1,NaN,0,0.0,0.908333,0
4,7.0,16,1,1,NaN,0,0.0,0.900000,0


In [14]:
dog_infos.tail()

,resultPosition,resultBtnDistance,resultSectionalTime,resultComment,resultRunTime,resultDogWeight,resultAdjustedTime,trapNumber,raceTime,raceDate,raceId,raceType,raceClass,raceDistance,raceGoing,raceWinTime,meetingId,trackName,dogId,numeratorSP,denominatorSP,commentScore,resultSectionalTime_missing,class_trajectory_5r,last_race_quality,has_open_race_exp,won_open_race,sprint_win_rate,had_recent_trial,hurdles_history_pct,history_comparability,has_ks_race
3383098,6.0,3.00,3.79,"VSAw,EP,Chl-1&Imp",30.820000,38.6,30.820000,4,12:18:00,2019-08-19,567861,Flat,A8,480.0,0.0,29.82000,354445,Perry Barr,99695,6.0,4.0,2.0,0,-3.0,6,0,0,NaN,0,0.0,1.0,0
3383099,3.0,0.20,3.68,"EP,Mid,SnLed-RnIn",29.780000,39.2,29.780000,4,13:44:00,2019-08-01,562048,Flat,A8,480.0,0.0,29.70000,353736,Perry Barr,99695,5.0,2.0,2.0,0,-3.0,6,0,0,NaN,0,0.0,1.0,0
3383100,5.0,1.25,3.68,"EP,Mid,Blk1,CrdRnIn",30.410000,39.4,30.410000,4,14:44:00,2019-07-28,560724,Flat,A8,480.0,0.0,29.88000,353611,Perry Barr,99695,3.0,1.0,0.0,0,-3.0,6,0,0,NaN,0,0.0,1.0,0
3383101,2.0,1.75,3.68,"EP,Mid,SnLed- 3/4",29.810000,39.5,29.810000,4,12:57:00,2019-07-18,557908,Flat,A8,480.0,0.0,29.67000,353265,Perry Barr,99695,6.0,1.0,1.0,0,NaN,9,0,0,NaN,0,0.0,NaN,0
3383102,3.0,0.30,NaN,"SAw,RanOn",28.698116,32.4,28.686835,3,18:12:00,2020-06-08,644703,Flat,D4,275.0,0.0,28.34907,362344,Doncaster,99972,3.0,1.0,0.0,1,NaN,9,0,0,NaN,0,0.0,NaN,0


In [6]:
model_df = dog_infos[dog_infos["resultSectionalTime_missing"]==0].copy()
sprint_races_df = dog_infos[dog_infos["resultSectionalTime_missing"]==1].copy()

model_df.reset_index(drop=True, inplace=True)
sprint_races_df.reset_index(drop=True, inplace=True)

model_df['raceClass'] = model_df['raceClass'].str.strip().str.upper()
sprint_races_df['raceClass'] = sprint_races_df['raceClass'].str.strip().str.upper()

In [78]:
model_df["raceClass"].value_counts()

raceClass
A5     341703
A4     338183
A6     322412
A3     308118
A7     283604
        ...  
M4          6
M5          6
GR          5
S15         4
HP          4
Name: count, Length: 71, dtype: int64

## 1.2 Race date features

We will use the race date to produce:

- `raceYear`: the year of the race

- `raceMonth`: month of the race

- `raceDayOfWeek`: day of the race week

- `raceDayOfYear`: day of the whole year

- `timeSinceLastRace`: time span between each races

This will give the model context on the time influence of the race.

In [ ]:
def time_since_last_race(dog_infos):
    dog_infos["raceDate"] = pd.to_datetime(dog_infos["raceDate"])
    
    dog_infos = dog_infos.sort_values(["dogId", "raceDate"])
    
    dog_infos["daysSinceLastRace"] = (
        dog_infos.groupby("dogId")["raceDate"]
        .transform(lambda x: x.diff().dt.days)
    )
    
    return dog_infos

In [ ]:
model_df["raceDate"] = pd.to_datetime(model_df["raceDate"], errors="coerce")
model_df["raceYear"] = model_df["raceDate"].dt.year
model_df["raceMonth"] = model_df["raceDate"].dt.month
model_df["raceDayOfWeek"] = model_df["raceDate"].dt.dayofweek
model_df["raceDayOfYear"] = model_df["raceDate"].dt.dayofyear

model_df = time_since_last_race(model_df)

## 1.3 SP features

- `spDecimal`: the division of numerator and denominator of SP

- `impliedProb`: implied market probability

In [ ]:
model_df["spDecimal"] = 1 + model_df["numeratorSP"].fillna(0) / model_df["denominatorSP"].replace(0, np.nan)
model_df["impliedProb"] = 1 / model_df["spDecimal"]

## 1.4 Time based features

We compute from all the time features the delta and speed features:

- `deltaWinTime`: difference between the winner and the dog result time

- `adjustedDeltaWinTime`: adjusted difference between the winner and the dog result time

- `runSpeed`: result time divided by the race distance

- `adjustedRunSpeed`: adjusted result time divided by the race distance

In [ ]:
model_df["deltaWinTime"] = model_df["resultRunTime"] - model_df["raceWinTime"]
model_df["adjustedDeltaWinTime"] = model_df["resultAdjustedTime"] - model_df["raceWinTime"]
model_df["runSpeed"] = model_df["raceDistance"] / model_df["resultRunTime"].replace(0, np.nan)
model_df["adjustedRunSpeed"] = model_df["raceDistance"] / model_df["resultAdjustedTime"].replace(0, np.nan)

## 1.5 Race class features

In [ ]:
model_df["raceClassFamily"] = model_df["raceClass"].str.extract(r"([A-Za-z]+)", expand=False).fillna("Unknown")
model_df["raceClassGrade"] = pd.to_numeric(
    model_df["raceClass"].str.extract(r"(\d+)", expand=False),
    errors="coerce",
)

## 1.6 Result comments features 

In [ ]:
comment_text = model_df["resultComment"].fillna("").str.lower()
model_df["commentLength"] = comment_text.str.len()
model_df["commentHasClear"] = comment_text.str.contains("clear|clrrun")
model_df["commentHasCrowded"] = comment_text.str.contains("crowd|crd")
model_df["commentHasBump"] = comment_text.str.contains("bmp|baulk")
model_df["commentHasEarlyPace"] = comment_text.str.contains("qaw|ep|led")
model_df["commentHasWide"] = comment_text.str.contains("wide|mid")

## 1.x ELO rating

## 1.x Early race position features

In [45]:
model_df.head()

,resultPosition,resultBtnDistance,resultSectionalTime,resultComment,resultRunTime,resultDogWeight,resultAdjustedTime,trapNumber,raceTime,raceDate,raceId,raceType,raceClass,raceDistance,raceGoing,raceWinTime,meetingId,trackName,dogId,numeratorSP,denominatorSP,commentScore,resultSectionalTime_missing,raceYear,raceMonth,raceDayOfWeek,raceDayOfYear,spDecimal,impliedProb,deltaWinTime,adjustedDeltaWinTime,runSpeed,adjustedRunSpeed,raceClassFamily,raceClassGrade,commentLength,commentHasClear,commentHasCrowded,commentHasBump,commentHasEarlyPace,commentHasWide
0,4.0,2.00,4.46,"Middle,EveryChance",28.73,34.1,28.73,5,21:18:00,2022-03-24,826826,Flat,A1,480.0,0.0,28.24,383435,Monmore,100211,5.0,4.000000,-1.0,0,2022,3,3,83,2.250000,0.444444,0.49,0.49,16.707275,16.707275,A,1.0,18,False,False,False,False,True
1,3.0,1.75,4.46,"CrdStt,Middle,RanOn",28.46,33.9,28.36,3,20:43:00,2022-03-12,823113,Flat,OR,480.0,-10.0,28.06,382921,Monmore,100211,7.0,1.000000,0.0,0,2022,3,5,71,8.000000,0.125000,0.40,0.30,16.865777,16.925247,OR,NaN,19,False,True,False,False,True
2,3.0,1.00,4.53,"Mid,FcdTCk&Crd1",28.72,33.9,28.72,3,16:48:00,2022-03-04,821053,Flat,A1,480.0,0.0,28.58,382798,Monmore,100211,11.0,1.675319,-1.0,0,2022,3,4,63,7.565913,0.132172,0.14,0.14,16.713092,16.713092,A,1.0,15,False,True,False,False,True
3,6.0,10.00,4.45,"Crd1,Chl&Crd&FellRunIn",28.98,34.1,28.85,4,18:17:00,2022-01-29,811419,Flat,A1,480.0,0.0,28.51,381677,Monmore,100211,1.0,1.000000,0.0,0,2022,1,5,29,2.000000,0.500000,0.47,0.34,16.563147,16.637782,A,1.0,22,False,True,False,False,False
4,2.0,10.00,4.45,"CrdStt,EP,Mid,RanOn",28.69,34.5,28.49,5,16:08:00,2022-01-21,809513,Flat,A1,480.0,-20.0,28.63,381477,Monmore,100211,11.0,1.675319,1.0,0,2022,1,4,21,7.565913,0.132172,0.06,-0.14,16.730568,16.848017,A,1.0,19,False,True,False,True,True


In [ ]:
def fetch_dog_past_races(dog_id, race_date, all_dogs_infos):
    dog_i_infos = all_dogs_infos[(all_dogs_infos["dogId"]==dog_id) & (all_dogs_infos["raceDate"]<race_date)]
    dog_i_infos.reset_index(drop=True, inplace=True)
    return dog_i_infos

def early_pos_race_dog(dog_infos):
    def rank_sectional(sec_times):
        sec_times_clean = np.where(np.isnan(sec_times), np.inf, sec_times)
        order = np.argsort(sec_times_clean)
        positions = np.empty_like(order)
        positions[order] = np.arange(1, len(sec_times_clean) + 1)
        return positions

    dog_infos["earlyPosition"] = (
        dog_infos.groupby("raceId")["resultSectionalTime"]
        .transform(rank_sectional)
    )
    dog_infos["deltaEarlyFinalPosition"] = (
        dog_infos["earlyPosition"] - dog_infos["resultPosition"]
    )



In [ ]:
early_pos_race_dog(model_df)

In [48]:
model_df["isBeginner"] = model_df["daysSinceLastRace"].isna().astype(int)

In [49]:
# EXAMPLE 
ex_row = model_df.iloc[3]
ex_dog_id = ex_row["dogId"]
ex_race_date = ex_row["raceDate"]
ex_dog_past_races = fetch_dog_past_races(ex_dog_id, ex_race_date, model_df)

In [50]:
ex_dog_past_races

,resultPosition,resultBtnDistance,resultSectionalTime,resultComment,resultRunTime,resultDogWeight,resultAdjustedTime,trapNumber,raceTime,raceDate,raceId,raceType,raceClass,raceDistance,raceGoing,raceWinTime,meetingId,trackName,dogId,numeratorSP,denominatorSP,commentScore,resultSectionalTime_missing,raceYear,raceMonth,raceDayOfWeek,raceDayOfYear,spDecimal,impliedProb,deltaWinTime,adjustedDeltaWinTime,runSpeed,adjustedRunSpeed,raceClassFamily,raceClassGrade,commentLength,commentHasClear,commentHasCrowded,commentHasBump,commentHasEarlyPace,commentHasWide,daysSinceLastRace,earlyPosition,deltaEarlyFinalPosition,isBeginner
0,3.0,4.75,4.52,"EP,Crd1",28.44,27.5,28.54,1,20:53:00,2016-03-22,203015,Flat,A6,450.0,10.0,27.79,310979,Poole,744,9.0,2.0,1.0,0,2016,3,1,82,5.5,0.181818,0.65,0.75,15.822785,15.767344,A,6.0,7,False,True,False,True,False,NaN,2,-1.0,1
1,1.0,0.00,4.46,"EP,Led 1/2",27.57,27.3,27.67,1,18:04:00,2016-03-27,204415,Flat,A7,450.0,10.0,27.57,311068,Poole,744,2.0,1.0,3.5,0,2016,3,6,87,3.0,0.333333,0.00,0.10,16.322089,16.263101,A,7.0,10,False,False,False,True,False,5.0,3,2.0,0
2,4.0,1.25,4.62,SAw,27.97,27.4,27.97,2,20:02:00,2016-04-03,206485,Flat,A6,450.0,0.0,27.52,311291,Poole,744,2.0,1.0,0.0,0,2016,4,6,94,3.0,0.333333,0.45,0.45,16.088666,16.088666,A,6.0,3,False,False,False,False,False,7.0,5,1.0,0


In [74]:
# Compute the resul position biased by the trap
stats = model_df.groupby(
    ['trackName','raceDistance',"trapNumber"]
)["resultPosition"].median().reset_index()

stats = stats.rename(columns={"resultPosition": "trapBiasResultPosition"})

model_df = model_df.merge(
    stats,
    on=['trackName', 'raceDistance', 'trapNumber'],
    how='left'
)

In [77]:
model_df

,resultPosition,resultBtnDistance,resultSectionalTime,resultComment,resultRunTime,resultDogWeight,resultAdjustedTime,trapNumber,raceTime,raceDate,raceId,raceType,raceClass,raceDistance,raceGoing,raceWinTime,meetingId,trackName,dogId,numeratorSP,denominatorSP,commentScore,resultSectionalTime_missing,raceYear,raceMonth,raceDayOfWeek,raceDayOfYear,spDecimal,impliedProb,deltaWinTime,adjustedDeltaWinTime,runSpeed,adjustedRunSpeed,raceClassFamily,raceClassGrade,commentLength,commentHasClear,commentHasCrowded,commentHasBump,commentHasEarlyPace,commentHasWide,daysSinceLastRace,earlyPosition,deltaEarlyFinalPosition,isBeginner,trapBiasResultPosition_x,trapBiasResultPosition_y,trapBiasResultPosition
0,3.0,4.75,4.52,"EP,Crd1",28.44,27.5,28.54,1,20:53:00,2016-03-22,203015,Flat,A6,450.0,10.0,27.79,310979,Poole,744,9.0,2.0,1.0,0,2016,3,1,82,5.50,0.181818,0.65,0.75,15.822785,15.767344,A,6.0,7,False,True,False,True,False,NaN,2,-1.0,1,3.0,3.373392,3.0
1,1.0,0.00,4.46,"EP,Led 1/2",27.57,27.3,27.67,1,18:04:00,2016-03-27,204415,Flat,A7,450.0,10.0,27.57,311068,Poole,744,2.0,1.0,3.5,0,2016,3,6,87,3.00,0.333333,0.00,0.10,16.322089,16.263101,A,7.0,10,False,False,False,True,False,5.0,3,2.0,0,3.0,3.373392,3.0
2,4.0,1.25,4.62,SAw,27.97,27.4,27.97,2,20:02:00,2016-04-03,206485,Flat,A6,450.0,0.0,27.52,311291,Poole,744,2.0,1.0,0.0,0,2016,4,6,94,3.00,0.333333,0.45,0.45,16.088666,16.088666,A,6.0,3,False,False,False,False,False,7.0,5,1.0,0,3.0,3.405871,3.0
3,4.0,1.75,4.56,"SAw,Crd3",28.07,28.1,28.07,2,20:52:00,2016-04-16,210756,Flat,A6,450.0,0.0,27.35,311861,Poole,744,3.0,1.0,-0.5,0,2016,4,5,107,4.00,0.250000,0.72,0.72,16.031350,16.031350,A,6.0,8,False,True,False,False,False,13.0,6,2.0,0,3.0,3.405871,3.0
4,1.0,0.00,4.54,"SAw,LedNearLine",27.86,27.8,27.86,1,22:22:00,2016-04-23,213059,Flat,A6,450.0,0.0,27.86,312112,Poole,744,3.0,1.0,0.0,0,2016,4,5,114,4.00,0.250000,0.00,0.00,16.152190,16.152190,A,6.0,15,False,False,False,True,False,7.0,6,5.0,0,3.0,3.373392,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2807442,1.0,0.00,4.22,"Mid,Ld1",30.28,34.1,30.08,3,16:44:00,2025-04-02,1119256,Flat,A5,500.0,-20.0,30.28,427665,Towcester,651853,5.0,4.0,1.5,0,2025,4,2,92,2.25,0.444444,0.00,-0.20,16.512550,16.622340,A,5.0,7,False,False,False,False,True,NaN,2,1.0,1,3.0,3.257178,3.0
2807443,1.0,0.00,3.67,"Railed,Led3",30.29,31.3,30.29,1,16:16:00,2025-03-30,1118688,Flat,A5,500.0,0.0,30.29,427589,Hove,651860,5.0,4.0,2.0,0,2025,3,6,89,2.25,0.444444,0.00,0.00,16.507098,16.507098,A,5.0,11,False,False,False,True,False,NaN,5,4.0,1,3.0,3.208468,3.0
2807444,6.0,0.10,4.38,CrdStart,30.74,26.1,30.74,2,11:52:00,2025-03-25,1117449,Flat,A6,500.0,0.0,29.94,427275,Sheffield,651896,6.0,1.0,0.0,0,2025,3,1,84,7.00,0.142857,0.80,0.80,16.265452,16.265452,A,6.0,8,False,True,False,False,False,NaN,3,-3.0,1,3.0,3.352033,3.0
2807445,5.0,10.00,3.88,"Rails,BumpedRunUp&1& 1/4",25.10,25.2,25.20,1,20:01:00,2025-04-02,1119551,Flat,A7,400.0,10.0,24.16,427694,Romford,652082,9.0,2.0,0.0,0,2025,4,2,92,5.50,0.181818,0.94,1.04,15.936255,15.873016,A,7.0,24,False,False,False,False,False,NaN,5,0.0,1,3.0,3.301800,3.0


In [52]:
model_df[(model_df["trackName"]=="Belle Vue") & (model_df["raceDistance"]==260)]

,resultPosition,resultBtnDistance,resultSectionalTime,resultComment,resultRunTime,resultDogWeight,resultAdjustedTime,trapNumber,raceTime,raceDate,raceId,raceType,raceClass,raceDistance,raceGoing,raceWinTime,meetingId,trackName,dogId,numeratorSP,denominatorSP,commentScore,resultSectionalTime_missing,raceYear,raceMonth,raceDayOfWeek,raceDayOfYear,spDecimal,impliedProb,deltaWinTime,adjustedDeltaWinTime,runSpeed,adjustedRunSpeed,raceClassFamily,raceClassGrade,commentLength,commentHasClear,commentHasCrowded,commentHasBump,commentHasEarlyPace,commentHasWide,daysSinceLastRace,earlyPosition,deltaEarlyFinalPosition,isBeginner


In [53]:
model_df[model_df["raceId"]==594843]

,resultPosition,resultBtnDistance,resultSectionalTime,resultComment,resultRunTime,resultDogWeight,resultAdjustedTime,trapNumber,raceTime,raceDate,raceId,raceType,raceClass,raceDistance,raceGoing,raceWinTime,meetingId,trackName,dogId,numeratorSP,denominatorSP,commentScore,resultSectionalTime_missing,raceYear,raceMonth,raceDayOfWeek,raceDayOfYear,spDecimal,impliedProb,deltaWinTime,adjustedDeltaWinTime,runSpeed,adjustedRunSpeed,raceClassFamily,raceClassGrade,commentLength,commentHasClear,commentHasCrowded,commentHasBump,commentHasEarlyPace,commentHasWide,daysSinceLastRace,earlyPosition,deltaEarlyFinalPosition,isBeginner
